# End-to-End Training Pipeline for Road Damage Detection
This notebook covers the entire pipeline: downloading the dataset, and training/evaluating three different object detection architectures (YOLOv8, YOLOv5, and RT-DETR).

## 1. Data Preparation
Downloading dataset from Roboflow and setting up the train/val/test split.

In [ ]:
!pip install roboflow ultralytics

In [ ]:
from roboflow import Roboflow

# Initialize Roboflow and download the dataset
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("road-damage").project("pothole-detection")
dataset = project.version(1).download("yolov8")

# Store the path to the dataset's yaml file so all models can use it
DATASET_YAML_PATH = f"{dataset.location}/data.yaml"

## 2. Train YOLOv8
Training the YOLOv8 architecture (CNN based).

In [ ]:
from ultralytics import YOLO

# Initialize Model
model_v8 = YOLO("yolov8n.pt")  # Load a pre-trained Nano model

# Training
results_v8 = model_v8.train(
    data=DATASET_YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.01,
    weight_decay=0.0005,
    project="results",
    name="yolov8_pothole"
)

# Evaluation on Test Set
metrics_v8 = model_v8.val(split="test")
print(f"YOLOv8 mAP50-95: {metrics_v8.box.map:.3f}")

## 3. Train YOLOv5 (Alternative)
Training the YOLOv5 architecture for comparison.

In [ ]:
# Initialize Model
model_v5 = YOLO("yolov5nu.pt") # Load a YOLOv5n architecture for comparison

# Training with different hyperparameters
results_v5 = model_v5.train(
    data=DATASET_YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.001, # Different learning rate
    optimizer="Adam", # Different optimizer
    project="results",
    name="yolo_alt_pothole"
)

# Evaluation on Test Set
metrics_v5 = model_v5.val(split="test")
print(f"YOLOv5 mAP50-95: {metrics_v5.box.map:.3f}")

## 4. Train RT-DETR (Vision Transformer)
Training a Transformer-based architecture to compare against the YOLO CNNs.

In [ ]:
from ultralytics import RTDETR

# Initialize Model
model_rtdetr = RTDETR("rtdetr-l.pt") # Load a ResNet50-backed RT-DETR model

# Training
results_rtdetr = model_rtdetr.train(
    data=DATASET_YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=8,
    lr0=0.0001,
    optimizer="AdamW",
    project="results",
    name="rtdetr_pothole"
)

# Evaluation on Test Set
metrics_rtdetr = model_rtdetr.val(split="test")
print(f"RT-DETR mAP50-95: {metrics_rtdetr.box.map:.3f}")